## Notebook for Experimenting Summarizer Generation

Author : Bagas Noor Fadhilah  
Student Number : 18223115  
  
---  
This notebook serves as an experiment playground for finetuning a model for summarization purpose and is part of a 3 stage pipeline. This model would be on the end of the pipeline or presumably the 3rd stage. The planned metrics that are going to be used are ROUGE, NLI, and if possible BERTScore. ROUGE serves as a lexical metric that is similar to EM (Exact Matching) and NLI would be monitoring the hallucinations that could be produced by the model. The model is going to be deployed via ngrok using their endpoint and the endpoint itself is going to be use from our local laptop that runs the python CLI script for demo.
  



```
{
  "input_text": "Summarize the following Steam reviews into a concise paragraph. Adhere strictly to the target aspects and their sentiments.\n\nTarget Aspects: Content: vast (positive) | Progression: rewarding (positive) | UI: dated (negative)\n\nReviews:\n- There is an insane amount of bosses, biomes, and gear to craft.\n- Progression feels earned every time you take down hardmode bosses.\n- Inventory management and UI feel cluttered and old-school.",
  "target_text": "Players praise the vast amount of craftable content and rewarding progression system, though the inventory UI feels dated and cluttered."
}
```

In [4]:
!nvidia-smi

Tue Nov 17 13:47:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!pip install evaluate
import evaluate

!pip install rouge_score



  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24985 sha256=6d39875bbe47bd2159f21a97b70a6cf9c3ab5ae2d1e6cc68fc25ab31acc792b7
  Stored in directory: /root/.cache/pip/wheels/44/af/da/5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge_score


In [6]:
import gc
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict, load_dataset
from evaluate import load
from peft import LoraConfig, PeftModel, get_peft_model
from sentence_transformers import CrossEncoder
from sklearn.model_selection import train_test_split
from transformers import (
  AutoModelForSeq2SeqLM,
  AutoTokenizer,
  DataCollatorForSeq2Seq,
  Seq2SeqTrainer,
  Seq2SeqTrainingArguments,
  set_seed,
)


In [7]:
SEED = 42
FLAN_MODEL = "google/flan-t5-base"
BART_MODEL = "facebook/bart-base"
BART_MODEL = "facebook/bart-large-cnn"
NLI_MODEL = "cross-encoder/nli-deberta-v3-base"
set_seed(42)

In [8]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
  print("GPU:", torch.cuda.get_device_name(0))
  device = torch.device("cuda")
else: device = torch.device("cpu")

CUDA available: True
GPU: Tesla T4


### Initial Dataset Loading

In [10]:
df = pd.read_csv("phase3dataset.csv")
df.head()

,summary_instruction,target
0,Summarize based on Reviews: Genshin Impact's o...,Genshin Impact offers a breathtaking and deepl...
1,Summarize based on Reviews: Zenless Zone Zero ...,"Zenless Zone Zero boasts stylish animations, e..."
2,Summarize based on Reviews: Elden Ring offers ...,Elden Ring offers an immersive world discovery...
3,Summarize based on Reviews: Cyberpunk 2077's N...,Cyberpunk 2077's night city offers an exceptio...


In [11]:
# TODO: split train_df into train and validation sets

y = df['target']

X_train, X_val, y_train, y_val = train_test_split(df, y, test_size=0.2, random_state=42)

### Utility Functions

In [12]:
NLI_model = CrossEncoder(NLI_MODEL)
FLAN_tokenizer = AutoTokenizer.from_pretrained(FLAN_MODEL)
BART_tokenizer = AutoTokenizer.from_pretrained(BART_MODEL)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


model.safetensors: reconstructing file:   0%|          |  0.00B /  738MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

... (12 outputs left out)

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [13]:
tokenized_sample = BART_tokenizer(df['summary_instruction'][0])

In [14]:
tokenized_sample

{'input_ids': [0, 38182, 3916, 2072, 716, 15, 35769, 35, 272, 1290, 11040, 14052, 18, 490, 12, 8331, 6942, 16, 24206, 8, 4814, 24111, 420, 358, 92, 976, 6, 600, 5796, 251, 21459, 396, 41, 2171, 34745, 27793, 64, 2128, 619, 35138, 4, 20, 43260, 25440, 467, 1189, 19223, 10314, 8, 4008, 10597, 15, 8544, 248, 5054, 13171, 6, 6437, 472, 7, 12371, 688, 9, 36096, 396, 10, 881, 33927, 2125, 4, 30456, 20234, 619, 20635, 12293, 8, 20666, 19, 45033, 11012, 6, 53, 5, 1830, 2280, 7014, 12, 261, 1905, 16, 23455, 3741, 25339, 148, 11138, 45999, 15798, 4, 20, 30062, 620, 7085, 19834, 14092, 13, 349, 11693, 1226, 16, 14528, 232, 12, 4684, 6, 14478, 1295, 847, 22270, 8, 25771, 25783, 88, 3722, 5586, 4, 20, 821, 22657, 31373, 467, 16, 5342, 16887, 13, 1804, 3768, 6, 53, 5, 4876, 12773, 1189, 29216, 8, 23477, 13, 614, 12, 4182, 1397, 472, 4, 944, 12, 1517, 30295, 14749, 2653, 11166, 9393, 8, 223, 428, 8435, 6, 25, 3918, 964, 1395, 490, 41125, 6, 1067, 7, 762, 47807, 6, 50, 3316, 232, 43491, 561, 10116, 28

In [15]:
BART_tokenizer.decode(tokenized_sample['input_ids'])

"<s>Summarize based on Reviews: Genshin Impact's open-world exploration is breathtaking and deeply immersive across every new region, though traveling long distances without an active stamina buff can sometimes feel tedious. The artifact grinding system remains utterly frustrating and heavily dependent on unfair RNG layers, forcing players to dump weeks of resin without a single usable piece. Combat mechanics feel remarkably fluid and responsive with elemental reactions, but the mobile camera lock-on feature is notoriously clunky during crowded Abyss encounters. The orchestral soundtrack composed for each distinct nation is genuinely world-class, elevating cutscenes and ambient roaming into emotional highlights. The gacha pity system is fairly predictable for limited characters, but the weapon banner remains predatory and punishing for low-spend players. Co-op multiplayer functionality feels severely restricted and underbaked, as visiting friends cannot open chests, talk to key NPCs, o

In [16]:
# max_length = 512

def tok_func(batch, tokenizer: AutoTokenizer):
  context = tokenizer(
      text=batch["summary_instruction"],
      truncation=True,
      padding=False,
      # max_length=max_length
  )
  summary = tokenizer(
      text=batch["target"],
      truncation=True,
      padding=False
  )
  context['labels'] = summary['input_ids']
  return context

In [17]:
rouge_metric = evaluate.load("rouge")

def compute_NLI(premises, decoded_labels):
  scores = NLI_model.predict(
    list(zip(premises, decoded_labels)),
    apply_softmax=True,
    batch_size=1
  ) # Return logits with shape of (1,labels)

  NLI_score = (float(scores[:,1].mean()))

  return NLI_score

def compute_metric_FLAN(eval_pred):
  predictions, labels = eval_pred

  decoded_preds = FLAN_tokenizer.batch_decode(predictions, skip_special_tokens=True)
  labels = np.where(labels != -100, labels, FLAN_tokenizer.pad_token_id)
  decoded_labels = FLAN_tokenizer.batch_decode(labels, skip_special_tokens=True)

  # print(decoded_labels)
  # print(list(valid_ds['summary_instruction']))
  # print(decoded_preds)

  result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
  nli_score = compute_NLI(list(valid_ds['summary_instruction']), decoded_preds)

  metrics = {k: round(v * 100, 4) for k, v in result.items()}
  metrics['entailment'] = nli_score

  return metrics

### Dataset Preps

In [18]:
metrics = {
    
}

In [19]:
train_ds = Dataset.from_pandas(X_train)
valid_ds = Dataset.from_pandas(X_val)

In [20]:
tokenized_flan_dataset = DatasetDict({
    "train": train_ds.map(lambda b: tok_func(b, FLAN_tokenizer), batched=True, desc="Tokenizing train"),
    "valid": valid_ds.map(lambda b: tok_func(b, FLAN_tokenizer), batched=True, desc="Tokenizing validation")
})

tokenized_bart_dataset = DatasetDict({
    "train": train_ds.map(lambda b: tok_func(b, BART_tokenizer), batched=True, desc="Tokenizing train"),
    "valid": valid_ds.map(lambda b: tok_func(b, BART_tokenizer), batched=True, desc="Tokenzing validation")
})

Tokenizing train:   0%|          | 0/3 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/1 [00:00<?, ? examples/s]

Tokenizing train:   0%|          | 0/3 [00:00<?, ? examples/s]

Tokenzing validation:   0%|          | 0/1 [00:00<?, ? examples/s]

In [21]:
data_collator_flan = DataCollatorForSeq2Seq(tokenizer=FLAN_tokenizer)
data_collator_bart = DataCollatorForSeq2Seq(tokenizer=BART_tokenizer)

### Training

In [22]:
FLAN_model = AutoModelForSeq2SeqLM.from_pretrained(FLAN_MODEL)
# BART_model = AutoModelForSeq2SeqLM.from_pretrained(BART_MODEL)
# BART_CNN_model = AutoModelForSeq2SeqLM.from_pretrained(BART_CNN_MODEL)

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

#### Zero Shot FLAN-T5

#### Fine Tune FLAN-T5

In [23]:
FLAN_training_args = Seq2SeqTrainingArguments(
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    # gradient_checkpointing=True,
    num_train_epochs=1,
    weight_decay=0.01,
    warmup_steps=0,
    predict_with_generate=True,
    generation_max_length=128,
    seed=SEED
)

In [24]:
trainer_FLAN = Seq2SeqTrainer(
    FLAN_model,
    args=FLAN_training_args,
    train_dataset=tokenized_flan_dataset['train'],
    eval_dataset=tokenized_flan_dataset['valid'], 
    data_collator=data_collator_flan,
    compute_metrics=compute_metric_FLAN
)

In [25]:
trainer_FLAN.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1, training_loss=1.952807903289795, metrics={'train_runtime': 22.909, 'train_samples_per_second': 0.131, 'train_steps_per_second': 0.044, 'total_flos': 2054272057344.0, 'train_loss': 1.952807903289795, 'epoch': 1.0})

In [26]:
trainer_FLAN.evaluate()

Training Loss,Validation Loss,Step,Rouge1,Rouge2,Rougel,Rougelsum,Entailment
No log,2.143492,1,23.529400,8.163300,19.607800,19.607800,0.993783


{'eval_loss': 2.143491506576538,
 'eval_rouge1': 23.5294,
 'eval_rouge2': 8.1633,
 'eval_rougeL': 19.6078,
 'eval_rougeLsum': 19.6078,
 'eval_entailment': 0.9937828183174133}

#### BART CNN Zero Shot

#### Fine Tune BART CNN

#### BART Zero Shot

#### Fine Tune BART

### Evaluations

Evaluate the overall performance of each proposed models (e.g. training time, val_loss, ROUGE, NLI)

In [ ]:
def evaluate_overall():
    pass

In [27]:
output = trainer_FLAN.model.generate(**FLAN_tokenizer(df['summary_instruction'][0], return_tensors='pt').to(device))

print(FLAN_tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (558 > 512). Running this sequence through the model will result in indexing errors


The game's open-world exploration is breathtaking and deeply immersive, but the artifact


### Model Deployment

In [28]:
!pip -q install pyngrok
!pip -q install python-dotenv
from pyngrok import ngrok
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("NGROK_PHASE_3_TOKEN")
ngrok.set_auth_token(api_key)

In [ ]:
deployed_model = trainer_FLAN

In [39]:
deployed_model

In [30]:
import time
import threading
from werkzeug.serving import make_server
from threading import Event
from flask import Flask, request, jsonify

In [56]:
# Clear ngrok endpoint usage
ngrok.kill()

In [52]:
app = Flask(__name__)
port = 5000

public_url = ngrok.connect(port).public_url
app.config["BASE_URL"] = public_url

@app.route("/")
def index():
  return "Hello from Colab phase 3!"

@app.route("/inference", methods=["POST"])
def inference():
  if not request.is_json:
    return jsonify({"msg": "the POST method must be JSON with the body containing 'content'"})
  else:
    content = request.json["content"]
    output = deployed_model.model.generate(**FLAN_tokenizer(content, return_tensors='pt').to(device))
    return jsonify({"msg": FLAN_tokenizer.decode(output[0], skip_special_tokens=True)})

class ThreadingServer(threading.Thread):
  def __init__(self, app, port):
    super().__init__()
    self.server = make_server("127.0.0.1", port, app)

  def run(self):
    print("Starting server")
    self.server.serve_forever()

  def shutdown(self):
    print("Shutting down server")
    self.server.shutdown()


In [53]:
print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:{port}\"")

 * ngrok tunnel "https://ongoing-vacant-savor.ngrok-free.dev" -> "http://127.0.0.1:5000"


In [54]:
server_thread = ThreadingServer(app, 5000)
server_thread.start() # Create new process in OS level so no blocking

Starting server


INFO:werkzeug:127.0.0.1 - - [17/Nov/2026 14:06:23] "POST /inference HTTP/1.1" 200 -


In [55]:
# Scram button for shutting down the server
server_thread.shutdown()
server_thread.join()

Shutting down server
